In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.LinearRegression import signature_helpers as sh


DATASET = 'FFPE_Human_Cervical_Cancer'
ADATA_PATH = f'{DATA_DIR}/Atera/{DATASET}/LinearRegression/hvgall_x_adata.h5ad'
CELL_TYPE_COL = 'cell_type'
T_CELL_TYPES = [
    'Naive & Memory T Cells',
    'Regulatory T Cells',
    'Cytotoxic T Cells',
    'Exhausted T Cells',
]

METHOD, PENALTY, STANDARDIZE = 'l1', 0.005, True
N_SUB = 50
LOWER, UPPER = 15, 85
DBVER = '2024.1.Hs'

# T-cell program signatures we PREDICT (as pseudo-gene targets).
TCELL_TARGETS = [
    'HALLMARK_IL2_STAT5_SIGNALING',
    'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY',
    'GOBP_T_CELL_MEDIATED_CYTOTOXICITY',
    'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE',
    'GOBP_CELL_KILLING',
    'GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY',
    'GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY',
]
NEG_CONTROL = [
    'HALLMARK_PANCREAS_BETA_CELLS',
    'HALLMARK_SPERMATOGENESIS',
]

# Cancer-associated signatures we ADD AS FACTORS (spatial-neighborhood predictors),
# split by cell type. PECAM1 (= CD31) is a one-gene vessel marker, its own set.
CANCER_FACTORS = [
    'HALLMARK_G2M_CHECKPOINT',
    'HALLMARK_E2F_TARGETS',
    'HALLMARK_ANGIOGENESIS',
    'HALLMARK_P53_PATHWAY',
    'HALLMARK_KRAS_SIGNALING_UP',
    'HALLMARK_KRAS_SIGNALING_DN',
    'HALLMARK_MYC_TARGETS_V1',
    'HALLMARK_GLYCOLYSIS',
    'HALLMARK_HYPOXIA',
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
    'HALLMARK_CHOLESTEROL_HOMEOSTASIS',
    'HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY',
    'KEGG_GLYCOLYSIS_GLUCONEOGENESIS',
    'KEGG_PENTOSE_PHOSPHATE_PATHWAY',
    'KEGG_CITRATE_CYCLE_TCA_CYCLE',
    'KEGG_ONE_CARBON_POOL_BY_FOLATE',
    'KEGG_PURINE_METABOLISM',
]
VESSEL = {'VESSEL_PECAM1': ['PECAM1']}

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')
print('\ncell_type value counts:')
print(adata.obs[CELL_TYPE_COL].value_counts())

In [ ]:
library = sh.load_msigdb(DBVER)
print('loaded', len(library), 'gene sets')

## T-cell TARGET signatures: membership, missing, present, factor-removed, remaining

In [ ]:
tcell_sets = sh.resolve_sets(TCELL_TARGETS + NEG_CONTROL, library, var,
                           label='T-cell targets + negative controls')
sh.report_factor_removal(adata, tcell_sets)

## Cancer FACTOR signatures: membership, missing, present (NOT predicted, so no factor-removal)

In [ ]:
cancer_sets = sh.resolve_sets(CANCER_FACTORS, library, var, label='cancer factor signatures')
# vessel marker (CD31 == PECAM1) as its own 1-gene factor set
cancer_sets['VESSEL_PECAM1'] = [g for g in VESSEL['VESSEL_PECAM1'] if g in var]
print('\nVESSEL_PECAM1 present in data:', cancer_sets['VESSEL_PECAM1'])

## Add the cancer signatures as spatial-neighborhood FACTORS

Each becomes `{cell_type}-{SET}` custom-factor columns (the diffused gene-set expression
contributed by each neighboring cell type), plus `{cell_type}-proximity` kernel-density
controls. These enter every T-cell target fit via `custom='all'`.

In [ ]:
sh.add_cancer_factors(adata, cancer_sets, annot_col=CELL_TYPE_COL, add_proximity=True)

## Predict each T-cell signature

For every signature, `run_signature` does three runs: (1) all T-cell types pooled, (2) all
T-cell types pooled with `cell_type` as an unpenalized covariate, (3) each T-cell type
individually. Each run prints the ranked table and the two ratio diagnostics
(highlighting the signature's own genes).

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_IL2_STAT5_SIGNALING', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_T_CELL_MEDIATED_CYTOTOXICITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_TNFA_SIGNALING_VIA_NFKB', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_INTERFERON_GAMMA_RESPONSE', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_INTERFERON_ALPHA_RESPONSE', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_CELL_KILLING', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

## Negative controls

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_PANCREAS_BETA_CELLS', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)

In [ ]:
_ = sh.run_signature(adata, 'HALLMARK_SPERMATOGENESIS', tcell_sets, cell_types=T_CELL_TYPES,
                     cell_type_col=CELL_TYPE_COL, method=METHOD, penalty=PENALTY,
                     standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER)